<a href="https://colab.research.google.com/github/ced-sys/SubTerra/blob/main/prithvi_mae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch einops timm

In [ ]:
import timm
print(timm.__version__)

1.0.20


In [ ]:
import warnings
import logging
import numpy as np
import torch
import torch.nn as nn
from einops import rearrange
from timm.layers import to_2tuple
from timm.models.vision_transformer import Block


In [ ]:
logger=logging.getLogger(__name__)

In [ ]:
def get_3d_sincos_embed(embed_dim, grid_size, add_cls_token=False):
  assert embed_dim % 16 == 0

  t_size, h_size, w_size=grid_size

  w_embed_dim=embed_dim // 16*6
  h_embed_dim=embed_dim // 16*6
  t_embed_dim=embed_dim // 16*4

  w_pos_embed=get_1d_sincos_pos_embed_from_grid(w_embed_dim, np.arange(w_size))
  h_pos_embed=get_1d_sincos_pos_embed_from_grid(h_embed_dim, np.arange(h_size))
  t_pos_embed=get_1d_sincos_pos_embed_from_grid(t_embed_dim, np.arange(t_size))

  w_pos_embed=np.tile(w_pos_embed, (t_size* h_size, 1))
  h_pos_embed=np.tile(np.repeat(h_pos_embed, w_size, axis=0), (t_size, 1))
  t_pos_embed=np.repeat(t_pos_embed, h_size* w_size, axis=0)

  pos_embed=np.concatenate((w_pos_embed, h_pos_embed, t_pos_embed), axis=1)

  if add_cls_token:
    pos_embed=np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
  return pos_embed



In [ ]:
def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
  """
  embed_dim: output dimesion for each position: a list of positions to be encoded: size(M,) out: (M,D)
  """
  if embed_dim % 2 !=0:
    raise ValueError("embed_dim must be even")

  omega=np.arange(embed_dim // 2, dtype=float)
  omega /= embed_dim /2.0
  omega=1.0 / 10000 **omega #(D/2)

  pos=pos.reshape(-1) #(M,)
  out=np.einsum("m,d->md", pos, omega) #(M, D/2), outer product

  emb_sin=np.sin(out) #(M, D/2)
  emb_cos=np.cos(out) #(M, D/2)

  emb=np.concatenate([emb_sin, emb_cos], axis=1) #(M, D)
  return emb


In [ ]:
def _get_1d_sincos_embed_from_grid_torch(embed_dim: int, pos: torch.Tensor):
  """Modified torch version of *get_1d_sincos_pos_embed_from_grid()^.

  embed_dim: output dimension for each position
  pos: a list of positions to be encoded: size(M,)- must be float dtype!
  out: (M, D)
  """
  assert embed_dim % 2==0
  assert pos.dtype in [torch.float32, torch.float16, torch.bfloat16]

  omega=torch.arange(embed_dim // 2, dtype=pos.dtype).to(pos.device)
  omega /=embed_dim /2.0
  omega=1.0 / 10000**omega #(D/2)

  pos=pos.reshape(-1) #(M,)
  out=torch.einsum("m, d->md", pos, omega) #(M, D/2), outer product

  emb_sin=torch.sin(out) #(M, D/2)
  emb_cos=torch.cos(out) #(M, D/2)

  emb=torch.cat([emb_sin, emb_cos], dim=1) #(M, D

  return emb

In [ ]:
def _init_weights(module):
  """Initialize weights"""
  if isinstance(module, nn.Linear):
    nn.init.xavier_uniform_(modeule.weight)
    if module.bias is not None:
      module.bias.data.zero_()
    elif isinstance(module. nn.LayerNorm):
      module.bias.data.zero_()
      module.weight.data.fill_(1.0)

In [ ]:
def _interpolate_pos_encoding(
    pos_embed: torch.Tensor,
    grid_size: tuple[int, int, int] | list[int],
    patch_size: tuple[int, int, int] | list[int],
    shape: tuple[int, int, int],
    embed_dim: int,
):

"""
Adapted form:
- transformers.models.cit.modeling_vit.ViTEmbeddings.interpolate_pos_encoding,
"""
t, h, w=shape
t_patches= t // patch_size[0]
h_patches=h // patch_size[1]
w_patches=w //patch_size[2]

if [t_patches, h_patches, w_patches]==grid_size:
  #No interpolation needed
  return pos_embed
if t_patches != grid_size[0]:
  #Re-compute pos embedding to handle changed  num_features
  new_grid_size=(t_patches, *grid_size[1:])
  new_pos_embed=get_3d_sincos_pos_embed(pos_embed.shape[-1], new_grid_size, add_cls_token=True)
  new_pos_embed=torch.from_numpy(new_pos_embed).float().unsqueeze(0)
else:
  new_grid